# Create a catalog & schema 

# With the help of autoloader, we will ingest data 


In [0]:
CATALOG="File_based"
SCHEMA="ecommerce"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.filelanding")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.schema")
          

In [0]:
spark.sql("DROP TABLE IF EXISTS file_based.ecommerce.pipeline_control")

In [0]:
spark.sql("DROP TABLE IF EXISTS file_based.ecommerce.bronze_layer")

In [0]:
spark.sql("DROP TABLE IF EXISTS file_based.ecommerce.silver_table")

In [0]:
spark.sql("DROP TABLE IF EXISTS file_based.ecommerce.bronze_layer")
spark.sql("DROP TABLE IF EXISTS file_based.ecommerce.silver_table")
spark.sql("DROP TABLE IF EXISTS file_based.ecommerce.gold_table")
 
dbutils.fs.rm("/Volumes/file_based/ecommerce/checkpoints/bronze/", recurse=True)
dbutils.fs.rm("/Volumes/file_based/ecommerce/schema/_schemas/",   recurse=True)
 

In [0]:

CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

File_landing_path = f"/Volumes/{CATALOG}/{SCHEMA}/filelanding"

Schema_inference_path = f"/Volumes/{CATALOG}/{SCHEMA}/schema"

In [0]:
files=dbutils.fs.ls(File_landing_path)
print(len(files))

for file in files:
    print(f" - {file.name}  ({file.size} bytes)")

In [0]:
spark.sql(f"create table if not exists {CATALOG}.{SCHEMA}.pipeline_control")

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_df=spark.readStream\
    .format("cloudFiles")\
    .option("cloudFiles.format","json")\
    .option("cloudFiles.schemaLocation",f"{Schema_inference_path}")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option("cloudFiles.inferColumnTypes","true")\
    .option("cloudFiles.useNotifications","false")\
    .load(f"{File_landing_path}")\
    .withColumn('source_file_name', F.col("_metadata.file_name"))\
    .withColumn("file_name", F.col("_metadata.file_name"))\
    .withColumn("file_size", F.col("_metadata.file_size"))\
    .withColumn("modified_time", F.col("_metadata.file_modification_time"))\
    .withColumn("_ingest_timestamp",F.current_timestamp())\
    .withColumn("_source_format", F.lit("json"))



In [0]:
bronze_df.writeStream.format("delta")\
         .outputMode("append")\
         .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze")\
         .trigger(availableNow=True)\
         .toTable(f"{CATALOG}.{SCHEMA}.bronze_layer")\
         .awaitTermination()    

print(f"Bronze load complete.")                  

In [0]:
bronze_count = spark.table(f"{CATALOG}.{SCHEMA}.bronze_layer").count()
print(f"Bronze load complete. Rows: {bronze_count}")

In [0]:
bronze_count = spark.table(f"{CATALOG}.{SCHEMA}.bronze_layer").count()
display(spark.table(f"{CATALOG}.{SCHEMA}.bronze_layer").limit(5))
print(f"\nTotal rows in Bronze: {bronze_count}")

In [0]:
bronze_df=spark.table(f"{CATALOG}.{SCHEMA}.Bronze_layer")
bronze_count=bronze_df.count()

duplicate_df=(bronze_df.groupBy("order_id","order_line_id").agg(F.count("*").alias("total_count")).filter(F.col("total_count")>1).count())

print(f"Total rows in Bronze      : {bronze_count}")
print(f"Business keys with dupes  : {duplicate_df}")
print()
print("Sample duplicates (same order_id appears more than once):")

In [0]:
from datetime import datetime

print(datetime.now())

In [0]:
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
SCHEMA_LOC_CSV= f"/Volumes/{CATALOG}/{SCHEMA}/schema/CSV"
SCHEMA_LOC_JSON= f"/Volumes/{CATALOG}/{SCHEMA}/schema/JSON"
File_landing_path = f"/Volumes/{CATALOG}/{SCHEMA}/filelanding"


In [0]:
from pyspark.sql import functions as F

In [0]:
def build_bronze_stream(fmt, schema_loc, checkpoint, glob_filter):
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",            fmt)
        .option("cloudFiles.schemaLocation",    schema_loc)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes",  "true")
        .option("cloudFiles.useNotifications",  "false")
        .option("pathGlobFilter",               glob_filter)
        .load(f"{File_landing_path}")
        .withColumn("file_name", F.col("_metadata.file_name"))\
        .withColumn("file_size", F.col("_metadata.file_size"))\
        .withColumn("modified_time", F.col("_metadata.file_modification_time"))
        .withColumn("_ingest_timestamp", F.current_timestamp())
        .withColumn("_source_format",    F.lit(fmt))
    )
 
# Stream 1 — CSV files
csv_query = (
    build_bronze_stream("csv", SCHEMA_LOC_CSV, CHECKPOINT_BASE, "*.csv")
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze/csv")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("file_based.ecommerce.bronze_layer")
)
 
# Stream 2 — JSON files
json_query = (
    build_bronze_stream("json", SCHEMA_LOC_JSON, CHECKPOINT_BASE, "*.json")
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze/json")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("file_based.ecommerce.bronze_layer")
)
 
# Wait for both streams to finish
csv_query.awaitTermination()
json_query.awaitTermination()
 
bronze_count = spark.table("file_based.ecommerce.bronze_layer").count()
print(f"Bronze load complete. Total rows: {bronze_count}")
print(f"Bronze schema:")
spark.table("file_based.ecommerce.bronze_layer").printSchema()

In [0]:
%sql 
select * from file_based.ecommerce.bronze_layer;